# Code Generation Report

Отчет по домену `code`: `HumanEval+`, `MBPP+`, `Functional Correctness`, `Pass@K` и efficiency.

Сравнение классов моделей:
- **M1** — крупные general-purpose LLM (DeepSeek, ChatGPT) через API
- **M2** — компактные специализированные code-модели (Qwen2.5-Coder, CodeGemma, DeepSeek-Coder-V2-Lite, CodeLlama) через Ollama

## 1. Setup And Data Sources

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent, PROJECT_ROOT.parent.parent):
    if (candidate / 'code').exists() and (candidate / 'shared').exists():
        PROJECT_ROOT = candidate
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / 'code') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'code'))
if str(PROJECT_ROOT / 'code' / 'notebooks') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'code' / 'notebooks'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from analysis_utils import (
    available_run_labels,
    ensure_expert_template,
    get_figures_dir,
    get_results_dir,
    load_candidate_metrics,
    load_records,
    load_sample_metrics,
    load_summary_metrics,
    model_family,
    pairwise_metric_deltas,
    reset_analysis_caches,
)

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 50)

reset_analysis_caches()
print(f'Results directory: {get_results_dir()}')
print(f'Available run labels: {available_run_labels()}')

In [ ]:
fc_summary = load_summary_metrics('fc', required=False)
passk_summary = load_summary_metrics('pass_k', required=False)
fc_samples = load_sample_metrics('fc', required=False)
fc_candidates = load_candidate_metrics('fc', required=False)
passk_candidates = load_candidate_metrics('pass_k', required=False)

for label, df in [('FC summary', fc_summary), ('Pass@K summary', passk_summary),
                   ('FC samples', fc_samples), ('FC candidates', fc_candidates)]:
    print(f'{label}: {len(df)} rows')

fc_summary['family'] = fc_summary['model_display_name'].map(model_family) if not fc_summary.empty else None
passk_summary['family'] = passk_summary['model_display_name'].map(model_family) if not passk_summary.empty else None
display(fc_summary.round(4) if not fc_summary.empty else pd.DataFrame({'status': ['No fc metrics yet']}))
display(passk_summary.round(4) if not passk_summary.empty else pd.DataFrame({'status': ['No pass_k metrics yet']}))

## 2. Dataset Overview

Объем данных, распределение по benchmark и характеристики промптов.

In [ ]:
samples_df, generations_df = load_records('fc')
if samples_df.empty:
    print('No raw records available.')
else:
    unique_samples = samples_df.drop_duplicates(subset=['benchmark', 'sample_id']).copy()
    overview = unique_samples.groupby('benchmark').agg(
        n_samples=('sample_id', 'nunique'),
        mean_prompt_len=('prompt_len', 'mean'),
        median_prompt_len=('prompt_len', 'median'),
    )
    display(overview.round(2))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.countplot(data=unique_samples, x='benchmark', ax=axes[0], palette='Set2')
    axes[0].set_title('Samples per Benchmark')
    axes[0].set_ylabel('Count')
    sns.histplot(data=unique_samples, x='prompt_len', hue='benchmark', bins=20, multiple='layer', ax=axes[1])
    axes[1].set_title('Prompt Length Distribution')
    axes[1].set_xlabel('Prompt length (chars)')
    plt.tight_layout()
    plt.savefig(get_figures_dir('fc') / 'report_01_dataset_overview.png', dpi=300, bbox_inches='tight')
    plt.show()

## 3. Main Results: Functional Correctness

Основные quality-метрики:  (pass@1) и . Сравнение M1 vs M2.

In [ ]:
if not fc_summary.empty:
    pass_cols = sorted([c for c in fc_summary.columns if c.startswith('pass@')],
                       key=lambda c: int(c.split('@')[1]))
    main = fc_summary[['model_display_name', 'family', 'benchmark', 'functional_correctness', *pass_cols]].copy()
    main = main.sort_values('functional_correctness', ascending=False)
    display(main.round(4))

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    order = main.sort_values('functional_correctness', ascending=False)['model_display_name'].tolist()
    palette = {name: ('#2196F3' if model_family(name) == 'M1' else '#FF9800') for name in order}

    sns.barplot(data=fc_summary, x='model_display_name', y='functional_correctness', hue='model_display_name',
                palette=palette, order=order, ax=axes[0], legend=False)
    axes[0].set_title('Functional Correctness (pass@1)')
    axes[0].set_ylabel('FC')
    axes[0].set_ylim(0, 1.05)
    axes[0].tick_params(axis='x', rotation=30)
    for bar in axes[0].patches:
        axes[0].annotate(f'{bar.get_height():.1%}', (bar.get_x() + bar.get_width()/2, bar.get_height()),
                        ha='center', va='bottom', fontsize=10)

    # M1 vs M2 grouped
    family_avg = fc_summary.groupby('family')['functional_correctness'].mean().reset_index()
    sns.barplot(data=family_avg, x='family', y='functional_correctness', hue='family',
                palette={'M1': '#2196F3', 'M2': '#FF9800'}, ax=axes[1], legend=False)
    axes[1].set_title('Average FC: M1 vs M2')
    axes[1].set_ylabel('FC')
    axes[1].set_ylim(0, 1.05)
    for bar in axes[1].patches:
        axes[1].annotate(f'{bar.get_height():.1%}', (bar.get_x() + bar.get_width()/2, bar.get_height()),
                        ha='center', va='bottom', fontsize=12)

    plt.tight_layout()
    plt.savefig(get_figures_dir('fc') / 'report_02_main_results.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('FC metrics are not available yet.')

In [ ]:
if not passk_summary.empty:
    pass_cols = sorted([c for c in passk_summary.columns if c.startswith('pass@')],
                       key=lambda c: int(c.split('@')[1]))
    display(passk_summary[['model_display_name', 'benchmark', *pass_cols]].round(4))

    long_df = passk_summary.melt(id_vars=['model_display_name', 'benchmark'], value_vars=pass_cols,
                                  var_name='metric', value_name='value')
    plt.figure(figsize=(12, 6))
    sns.barplot(data=long_df, x='metric', y='value', hue='model_display_name')
    plt.title('Pass@K Comparison')
    plt.ylabel('Score')
    plt.ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig(get_figures_dir('pass_k') / 'pass_at_k.png', dpi=150)
    plt.show()
else:
    print('Pass@K metrics are not available yet.')

## 4. Efficiency

Сравнение по latency, token usage и estimated cost. Trade-off между качеством и вычислительными затратами.

In [ ]:
if not fc_summary.empty:
    eff_cols = ['model_display_name', 'family', 'benchmark', 'Tinf', 'Tok', 'Cost', 'Eff_normalized']
    eff_df = fc_summary[[c for c in eff_cols if c in fc_summary.columns]].copy()
    display(eff_df.round(4))

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    order = fc_summary.sort_values('functional_correctness', ascending=False)['model_display_name'].tolist()
    palette = {name: ('#2196F3' if model_family(name) == 'M1' else '#FF9800') for name in order}

    sns.barplot(data=fc_summary, x='model_display_name', y='Tinf', hue='model_display_name',
                palette=palette, order=order, ax=axes[0], legend=False)
    axes[0].set_title('Inference Latency (ms)')
    axes[0].set_ylabel('ms')
    axes[0].tick_params(axis='x', rotation=30)

    sns.barplot(data=fc_summary, x='model_display_name', y='Tok', hue='model_display_name',
                palette=palette, order=order, ax=axes[1], legend=False)
    axes[1].set_title('Token Usage')
    axes[1].set_ylabel('tokens')
    axes[1].tick_params(axis='x', rotation=30)

    sns.barplot(data=fc_summary, x='model_display_name', y='Cost', hue='model_display_name',
                palette=palette, order=order, ax=axes[2], legend=False)
    axes[2].set_title('Estimated Cost (USD)')
    axes[2].set_ylabel('USD')
    axes[2].tick_params(axis='x', rotation=30)

    plt.tight_layout()
    plt.savefig(get_figures_dir('fc') / 'report_03_efficiency_components.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Quality vs Cost scatter
    fig, ax = plt.subplots(figsize=(10, 6))
    for _, row in fc_summary.iterrows():
        color = '#2196F3' if model_family(row['model_display_name']) == 'M1' else '#FF9800'
        ax.scatter(row['Tinf'], row['functional_correctness'], s=200, c=color, edgecolors='black', zorder=5)
        ax.annotate(row['model_display_name'], (row['Tinf'], row['functional_correctness']),
                   textcoords='offset points', xytext=(8, 5), fontsize=9)
    ax.set_xlabel('Average Latency (ms)')
    ax.set_ylabel('Functional Correctness')
    ax.set_title('Quality vs Speed Trade-off')
    ax.set_ylim(0, 1.05)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(color='#2196F3', label='M1 (API)'), Patch(color='#FF9800', label='M2 (Local)')],
              loc='lower right')
    plt.tight_layout()
    plt.savefig(get_figures_dir('fc') / 'report_04_quality_vs_speed.png', dpi=300, bbox_inches='tight')
    plt.show()

## 5. Hypothesis-Driven Comparison

Парное сравнение моделей на уровне sample с bootstrap confidence interval.

In [ ]:
from itertools import combinations

if not fc_samples.empty:
    rows = []
    for benchmark, group in fc_samples.groupby('benchmark'):
        pivot = group.pivot_table(index='sample_id', columns='model_display_name', values='first_hit', aggfunc='first')
        model_names = list(pivot.columns)
        if len(model_names) < 2:
            continue
        for left, right in combinations(model_names, 2):
            left_vals = pivot[left].fillna(False).astype(bool)
            right_vals = pivot[right].fillna(False).astype(bool)
            diff = float(left_vals.mean() - right_vals.mean())
            diffs = []
            pair_df = pd.DataFrame({'left': left_vals, 'right': right_vals})
            for _ in range(1000):
                sample = pair_df.sample(n=len(pair_df), replace=True)
                diffs.append(float(sample['left'].mean() - sample['right'].mean()))
            ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
            rows.append({
                'benchmark': benchmark,
                'left_model': left,
                'right_model': right,
                'pair': f'{left} vs {right}',
                'left_accuracy': float(left_vals.mean()),
                'right_accuracy': float(right_vals.mean()),
                'accuracy_diff': diff,
                'ci_low': float(ci_low),
                'ci_high': float(ci_high),
                'significant': not (ci_low <= 0 <= ci_high),
                'left_only_correct': int((left_vals & ~right_vals).sum()),
                'right_only_correct': int((~left_vals & right_vals).sum()),
            })
    hyp_df = pd.DataFrame(rows)

    # Show only M1 vs best M2 and M2 vs M2 pairs
    display(hyp_df[['benchmark', 'left_model', 'right_model', 'accuracy_diff', 'ci_low', 'ci_high', 'significant']].round(4))

    if not hyp_df.empty:
        # Focus on key pairs
        fig, ax = plt.subplots(figsize=(14, max(6, len(hyp_df) * 0.5)))
        hyp_sorted = hyp_df.sort_values('accuracy_diff', ascending=True)
        y_pos = range(len(hyp_sorted))
        colors = ['#4CAF50' if row['significant'] else '#9E9E9E' for _, row in hyp_sorted.iterrows()]
        ax.barh(list(y_pos), hyp_sorted['accuracy_diff'], color=colors, height=0.6)
        ax.set_yticks(list(y_pos))
        ax.set_yticklabels(hyp_sorted['pair'], fontsize=9)
        ax.axvline(0, color='black', linewidth=1, linestyle='--')
        ax.set_xlabel('Accuracy Difference (left - right)')
        ax.set_title('Pairwise FC Differences (green = significant at 95% CI)')
        plt.tight_layout()
        plt.savefig(get_figures_dir('fc') / 'report_05_hypothesis.png', dpi=300, bbox_inches='tight')
        plt.show()
else:
    print('No sample-level metrics available for hypothesis testing.')

## 6. Error Analysis

Разбор sample-level ошибок: типы промахов, overlap между моделями M1 и M2.

In [ ]:
if not fc_candidates.empty:
    def categorize_error(row):
        if row.get('functional_correctness'):
            return 'correct'
        error_type = row.get('error_type', '')
        if error_type == 'empty_code':
            return 'empty_code'
        if error_type == 'syntax_error':
            return 'syntax_error'
        if error_type == 'timeout':
            return 'timeout'
        return 'failed_tests'

    # Only first candidate per sample for fc mode
    first_candidates = fc_candidates[fc_candidates['candidate_index'] == 1].copy()
    first_candidates['error_category'] = first_candidates.apply(categorize_error, axis=1)
    first_candidates['family'] = first_candidates['model_display_name'].map(model_family)

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    # Error type distribution
    failures = first_candidates[first_candidates['error_category'] != 'correct']
    if not failures.empty:
        sns.countplot(data=failures, x='error_category', hue='model_display_name', ax=axes[0])
        axes[0].set_title('Error Categories by Model')
        axes[0].tick_params(axis='x', rotation=20)
        axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    else:
        axes[0].text(0.5, 0.5, 'No failures', ha='center', va='center', transform=axes[0].transAxes)

    # Error type distribution M1 vs M2
    if not failures.empty:
        error_counts = failures.groupby(['family', 'error_category']).size().reset_index(name='count')
        sns.barplot(data=error_counts, x='error_category', y='count', hue='family',
                   palette={'M1': '#2196F3', 'M2': '#FF9800'}, ax=axes[1])
        axes[1].set_title('Error Categories: M1 vs M2')
        axes[1].tick_params(axis='x', rotation=20)

    plt.tight_layout()
    plt.savefig(get_figures_dir('fc') / 'report_06_error_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Error rate summary
    error_summary = first_candidates.groupby('model_display_name').agg(
        total=('sample_id', 'count'),
        correct=('functional_correctness', 'sum'),
        empty=('error_category', lambda x: (x == 'empty_code').sum()),
        syntax=('error_category', lambda x: (x == 'syntax_error').sum()),
        timeout=('error_category', lambda x: (x == 'timeout').sum()),
        wrong_answer=('error_category', lambda x: (x == 'failed_tests').sum()),
    ).reset_index()
    error_summary['error_rate'] = 1 - error_summary['correct'] / error_summary['total']
    display(error_summary.round(4))
else:
    print('No candidate metrics available for error analysis.')

In [ ]:
# Overlap analysis: which samples does M1 solve that M2 cannot and vice versa
if not fc_samples.empty:
    pivot = fc_samples.pivot_table(index='sample_id', columns='model_display_name', values='first_hit', aggfunc='first')
    pivot = pivot.fillna(False).astype(bool)

    m1_models = [c for c in pivot.columns if model_family(c) == 'M1']
    m2_models = [c for c in pivot.columns if model_family(c) == 'M2']

    if m1_models and m2_models:
        m1_any = pivot[m1_models].any(axis=1)
        m2_any = pivot[m2_models].any(axis=1)
        best_m2_name = fc_summary.loc[fc_summary['family'] == 'M2', 'functional_correctness'].idxmax()
        best_m2_name = fc_summary.loc[best_m2_name, 'model_display_name'] if best_m2_name is not None else m2_models[0]

        overlap_data = {
            'Both M1 & M2 solve': int((m1_any & m2_any).sum()),
            'Only M1 solves': int((m1_any & ~m2_any).sum()),
            'Only M2 solves': int((~m1_any & m2_any).sum()),
            'Neither solves': int((~m1_any & ~m2_any).sum()),
        }
        overlap_df = pd.DataFrame(list(overlap_data.items()), columns=['Category', 'Count'])
        display(overlap_df)

        fig, ax = plt.subplots(figsize=(8, 6))
        colors = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
        ax.pie(overlap_df['Count'], labels=overlap_df['Category'], colors=colors,
               autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
        ax.set_title('Sample Overlap: M1 vs M2 (any model in class)')
        plt.tight_layout()
        plt.savefig(get_figures_dir('fc') / 'report_07_overlap.png', dpi=300, bbox_inches='tight')
        plt.show()

## 7. Expert Score Placeholder

Если экспертная разметка заполнена, здесь агрегируются .

In [ ]:
expert_path = ensure_expert_template('fc')
expert_df = pd.read_csv(expert_path) if Path(expert_path).exists() else pd.DataFrame()
score_cols = ['completeness', 'efficiency', 'readability']
for col in score_cols:
    if col not in expert_df.columns:
        expert_df[col] = pd.NA

if not expert_df.empty and 'model_display_name' not in expert_df.columns:
    display_map = fc_samples[['model_name', 'model_display_name']].drop_duplicates()
    expert_df = expert_df.merge(display_map, on='model_name', how='left')
    expert_df['model_display_name'] = expert_df['model_display_name'].fillna(expert_df['model_name'])

expert_df['is_scored'] = expert_df[score_cols].notna().all(axis=1) if not expert_df.empty else pd.Series(dtype=bool)
display(expert_df.head())

if not expert_df.empty:
    coverage = (
        expert_df.groupby(['benchmark', 'model_display_name'])['is_scored']
        .agg(['sum', 'count'])
        .reset_index()
        .rename(columns={'sum': 'scored_rows', 'count': 'total_rows'})
    )
    coverage['coverage'] = coverage['scored_rows'] / coverage['total_rows'].clip(lower=1)
    display(coverage.round(3))

    plt.figure(figsize=(8, 5))
    sns.barplot(data=coverage, x='benchmark', y='coverage', hue='model_display_name')
    plt.title('Expert Annotation Coverage')
    plt.ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig(get_figures_dir('fc') / 'report_08_expert_coverage.png', dpi=300, bbox_inches='tight')
    plt.show()

print(f'Expert template: {expert_path}')

## 8. Conclusions

Итоговая сводка: качество, скорость, стоимость. Ранжирование моделей.

In [ ]:
if not fc_summary.empty:
    conclusion = fc_summary[['model_display_name', 'family', 'benchmark', 'functional_correctness', 'Tinf', 'Tok', 'Cost']].copy()
    conclusion = conclusion.sort_values('functional_correctness', ascending=False)
    conclusion = conclusion.rename(columns={
        'functional_correctness': 'FC',
        'Tinf': 'Latency_ms',
        'Tok': 'Avg_Tokens',
        'Cost': 'Cost_USD',
    })
    display(conclusion.round(4))

    best_m1 = fc_summary[fc_summary['family'] == 'M1']['functional_correctness'].max()
    best_m2 = fc_summary[fc_summary['family'] == 'M2']['functional_correctness'].max()
    best_m2_name = fc_summary.loc[
        (fc_summary['family'] == 'M2') & (fc_summary['functional_correctness'] == best_m2),
        'model_display_name'
    ].iloc[0]
    gap = best_m1 - best_m2

    print()
    print(f'Best M1 FC: {best_m1:.1%}')
    print(f'Best M2 FC: {best_m2:.1%} ({best_m2_name})')
    print(f'Gap M1 - M2: {gap:.1%}')
    print()
    print('M2 models are free (local inference) vs M1 API cost.')
    print(f'Best M2 achieves {best_m2/best_m1:.0%} of best M1 performance.')
else:
    print('No FC metrics available for conclusions.')